# 01 · How a fraud score learns

**Project goal.** Give a transaction a risk score and use a threshold to decide whether to review it. In this lesson you can follow one prediction by hand, then check precisely how the fraud label changes the next prediction.

Start here in a fresh Colab CPU runtime. Work through each prediction before running its cell. The numbers below describe invented transactions. Notebook 02 uses the ULB dataset.

## Follow a single weight
A hidden neuron produces $h=0.5$. Its outgoing weight starts at zero. We choose a bias that gives an initial fraud score of 0.20. For a fraud label $y=1$, the binary cross-entropy is $L=-\log p$. Predict the sign of the gradient and the direction of the next score.

In [1]:
import torch
from torch import nn

# Start with one hidden neuron's output. Only its outgoing weight will change in this example.
h = torch.tensor(0.5)
weight = nn.Parameter(torch.tensor(0.0))
# With weight = 0, this bias gives sigmoid(logit) = 0.20 before learning.
bias = torch.logit(torch.tensor(0.2))
target = torch.tensor(1.0)
logit = h * weight + bias
# For a fraud label of 1, the loss is -log(score). The function takes the raw logit.
loss = nn.functional.binary_cross_entropy_with_logits(logit, target)
# Backpropagation computes the derivative of the loss with respect to each trainable weight.
loss.backward()
# The chain rule gives (score - label) * h. Here the gradient is -0.40.
manual = (logit.detach().sigmoid() - target) * h
torch.testing.assert_close(weight.grad, manual)
print(f"Initial score {logit.sigmoid().item():.3f}, gradient {weight.grad.item():.3f}")
# SGD subtracts learning_rate * gradient, so this weight rises by 0.04.
optimizer = torch.optim.SGD([weight], lr=0.1)
# Apply the optimizer's update using the gradients currently stored on the parameters.
optimizer.step()
print(f"New weight {weight.item():.3f}, new score {(weight * h + bias).sigmoid().item():.3f}")

Initial score 0.200, gradient -0.400
New weight 0.040, new score 0.203


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
# Mount My Drive so this Colab runtime can read files stored in your account.
drive.mount('/content/drive')

Mounted at /content/drive


## Backpropagate through a hidden layer
Each row of $X$ holds two binary signals. A row is labeled 1 when exactly one signal is active. This XOR rule gives the hidden layer a reason to learn a nonlinear transformation.

For a batch of $n$ rows, compute $H=\tanh(XW_1+b_1)$, $z=HW_2+b_2$ and $p=\sigma(z)$. Mean binary cross-entropy gives $D=(p-y)/n$. Multiplying by $W_2^T$ sends that error back to the hidden units. Multiplying elementwise by $1-H^2$ accounts for the slope of tanh.

In [3]:
torch.manual_seed(42)
# Each row is a toy transaction. The label is 1 when exactly one of its two signals is active.
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])
model = nn.Sequential(nn.Linear(2, 4), nn.Tanh(), nn.Linear(4, 1))
# Four rows × two inputs become four rows × four hidden activations.
hidden = model[:2](X)
# Combine the four activations into one raw score for each row.
logits = model[2](hidden)
criterion = nn.BCEWithLogitsLoss()
criterion(logits, y).backward()

# nn.Linear stores its weights as [outputs, inputs].
with torch.no_grad():
    # The loss averages four rows, so divide each row's output error by four.
    delta = (logits.sigmoid() - y) / len(X)
    # Send each output error back through the weights, then multiply by the slope of tanh.
    hidden_delta = (delta @ model[2].weight) * (1 - hidden.square())
    torch.testing.assert_close(model[2].weight.grad, delta.T @ hidden)
    torch.testing.assert_close(model[0].weight.grad, hidden_delta.T @ X)
    # A bias is shared by all rows, so its gradient adds up their contributions.
    torch.testing.assert_close(model[2].bias.grad, delta.sum(0))
    torch.testing.assert_close(model[0].bias.grad, hidden_delta.sum(0))
print("All manual gradients match autograd.")

All manual gradients match autograd.


In [4]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
for epoch in range(201):
    # Clear gradients from the previous update. PyTorch adds new gradients to any already stored.
    optimizer.zero_grad()
    loss = criterion(model(X), y)
    # Backpropagation computes the derivative of the loss with respect to each trainable weight.
    loss.backward()
    # Apply the optimizer's update using the gradients currently stored on the parameters.
    optimizer.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | mean loss {loss.item():.4f}")

with torch.no_grad():
    print("signal A, signal B, label, score")
    # Read each row as two input signals, the true label, and the learned score.
    print(torch.cat([X, y, model(X).sigmoid()], dim=1))

Epoch   0 | mean loss 0.7598
Epoch  50 | mean loss 0.2793
Epoch 100 | mean loss 0.0602
Epoch 150 | mean loss 0.0295
Epoch 200 | mean loss 0.0186
signal A, signal B, label, score
tensor([[0.0000, 0.0000, 0.0000, 0.0018],
        [0.0000, 1.0000, 1.0000, 0.9801],
        [1.0000, 0.0000, 1.0000, 0.9809],
        [1.0000, 1.0000, 0.0000, 0.0321]])


## Checkpoint before the real dataset
1. Reproduce the first weight update by hand from $p=0.2$, $y=1$, $h=0.5$ and learning rate $0.1$. Explain why a small change in the weight changes the score.
2. Why is the error divided by four in the XOR batch? What would happen to the gradients if we accumulated them across epochs without clearing them?
3. Predict what happens if you replace tanh with the identity function and retrain from initialization. Explain your prediction using the shape of the decision boundary.

Continue to notebook 02 when you can explain the forward pass, the gradient and the update in your own words.